In [ ]:
# spark seassion
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, round as spark_round, row_number
from pyspark.sql.window import Window
import pandas as pd

# Membuat SparkSession — "local[*]" berarti gunakan seluruh core CPU yang tersedia di VM
spark = SparkSession.builder \
    .appName("Tugas5-2505060041zuyinamirkham") \
    .master("local[*]") \
    .getOrCreate()

# Mengurangi banyaknya pesan log teknis agar output lebih bersih
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession berhasil dibuat!")
print("Versi Spark:", spark.version)

# 1) Baca CSV dari HDFS, lalu tambah kolom pendapatan
df_transaksi = (
    spark.read
    .csv("hdfs://localhost:9000/user/irkham/tugas5/transaksi_tugas5.csv",
         header=True, inferSchema=True)
    .withColumn("pendapatan", col("unit_terjual") * col("harga_satuan"))
)

# 2) Buat df_target dari dictionary
data_target_cabang = {
    "kota": ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"],
    "target_bulanan": [45000000, 60000000, 55000000, 40000000, 30000000],
    "pic_cabang": ["Rani", "Joko", "Sari", "Bayu", "Fitri"],
}
df_target = spark.createDataFrame(pd.DataFrame(data_target_cabang))

# Cek hasil
print("Tipe objek:", type(df_transaksi))
df_transaksi.printSchema()
df_transaksi.show(10)
print("Jumlah baris:", df_transaksi.count())

df_target.show()

SparkSession berhasil dibuat!
Versi Spark: 3.5.9
Tipe objek: <class 'pyspark.sql.dataframe.DataFrame'>
root
 |-- order_id: string (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- pendapatan: integer (nullable = true)

+--------+--------------------+----------+------------+------------+----------+
|order_id|            kategori|      kota|unit_terjual|harga_satuan|pendapatan|
+--------+--------------------+----------+------------+------------+----------+
|   TRX-0|   Makanan & Minuman| Purworejo|           8|       75000|    600000|
|   TRX-1|          Elektronik|      Solo|           9|       75000|    675000|
|   TRX-2|Kesehatan & Kecan...|      Solo|           6|      100000|    600000|
|   TRX-3|             Fashion|Yogyakarta|           6|      100000|    600000|
|   TRX-4|          Elektronik|Yogyakarta|           2|       75000|    150000|
| 

In [ ]:
# A.Join & Perbandingan Target
df_ringkasan_kota = df_transaksi.groupBy("kota").agg(
    spark_sum("pendapatan").alias("total_pendapatan")
)

df_pencapaian = df_ringkasan_kota.join(df_target, on="kota", how="inner") \
    .withColumn("pencapaian_persen", (col("total_pendapatan") / col("target_bulanan")) * 100) \
    .orderBy(col("pencapaian_persen").desc())

df_pencapaian.show()

+----------+----------------+--------------+----------+------------------+
|      kota|total_pendapatan|target_bulanan|pic_cabang| pencapaian_persen|
+----------+----------------+--------------+----------+------------------+
| Purworejo|        45650000|      30000000|     Fitri|152.16666666666669|
|      Solo|        33475000|      40000000|      Bayu|           83.6875|
|Yogyakarta|        47275000|      60000000|      Joko| 78.79166666666667|
|  Magelang|        31650000|      45000000|      Rani| 70.33333333333334|
|  Semarang|        38175000|      55000000|      Sari|  69.4090909090909|
+----------+----------------+--------------+----------+------------------+



In [ ]:
# B.Window Function — Kategori Terlaris per Kota
df_kategori_kota = df_transaksi.groupBy("kota", "kategori").agg(
    spark_sum("pendapatan").alias("total_pendapatan")
)

window_kategori = Window.partitionBy("kota").orderBy(col("total_pendapatan").desc())

df_kategori = df_kategori_kota.withColumn("peringkat", row_number().over(window_kategori)) \
    .filter(col("peringkat") == 1) \
    .orderBy(col("kota").desc())

df_kategori.show()

+----------+--------------------+----------------+---------+
|      kota|            kategori|total_pendapatan|peringkat|
+----------+--------------------+----------------+---------+
|Yogyakarta|             Fashion|        13325000|        1|
|      Solo|Kesehatan & Kecan...|         8425000|        1|
|  Semarang|        Rumah Tangga|        11125000|        1|
| Purworejo|Kesehatan & Kecan...|        10075000|        1|
|  Magelang|Kesehatan & Kecan...|         7275000|        1|
+----------+--------------------+----------------+---------+



In [ ]:
df_transaksi.createOrReplaceTempView("transaksi")
df_target.createOrReplaceTempView("target_cabang")

df_transaksi_terbanyak = spark.sql("""
    SELECT t.kota, g.pic_cabang, COUNT(t.order_id) AS jumlah_transaksi
    FROM transaksi t
    JOIN target_cabang g ON t.kota = g.kota
    GROUP BY t.kota, g.pic_cabang
    ORDER BY jumlah_transaksi DESC
""")

df_transaksi_terbanyak.show()

+----------+----------+----------------+
|      kota|pic_cabang|jumlah_transaksi|
+----------+----------+----------------+
| Purworejo|     Fitri|             116|
|Yogyakarta|      Joko|             110|
|      Solo|      Bayu|              95|
|  Semarang|      Sari|              93|
|  Magelang|      Rani|              86|
+----------+----------+----------------+



berdasarkan hasil bagian A dan B, cabang mana yang berkinerja paling baik dan cabang mana yang paling perlu perhatian manajemen? Sertakan angka-angka pendukung dari hasil analisis kalian, bukan opini tanpa dasar data.
Jawab:
Cabang Purworejo dengan penanggung jawab Fitri mencatatkan kinerja terbaik dengan tingkat pencapaian target mencapai 152,17%. Cabang Purworejo berhasil membukukan total pendapatan sebesar Rp45.650.000 dari target bulanan Rp30.000.000 dengan jumlah transaksi tertinggi sebanyak 116 transaksi. Kategori penyokong utama pendapatan di Purworejo adalah Kesehatan & Kecantikan yang menyumbang Rp10.075.000.

Sementara itu, cabang Semarang dengan penanggung jawab Sari menjadi cabang yang paling memerlukan perhatian khusus manajemen karena mencatatkan persentase pencapaian target terendah, yaitu sebesar 69,41%. Cabang Semarang baru memperoleh pendapatan sebesar Rp38.175.000 dari target tinggi sebesar Rp55.000.000, atau kekurangan Rp16.825.000, dengan total 93 transaksi. Berdasarkan Bagian B, pendapatan Semarang bertumpu pada kategori Rumah Tangga (Rp11.125.000), sehingga manajemen perlu mendorong kategori lain agar volume transaksi meningkat.

In [ ]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
